# Pillar-0 spatial audit: visual QA

CPU-only preprocessing check for a report-only knee study. This notebook reconstructs the three acquired planes from DICOM geometry and displays central orthogonal views. It does not load a model, train, or read official labels. Review the images before running the separate frozen-feature audit.


In [ ]:
### Cell 1 — locate the dataset and load its tables

from pathlib import Path
import random

import numpy as np
import pandas as pd

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# Kaggle may expose competition data in either location.
data_candidates = [
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
]

DATA = next((path for path in data_candidates if path.is_dir()), None)

if DATA is None:
    raise FileNotFoundError(
        "Competition data was not found. Attach "
        "'rsna-knee-abnormality-detection' to this notebook."
    )

csv_paths = sorted(DATA.glob("*.csv"))

print("Dataset directory:", DATA)
print("CSV files:", [path.name for path in csv_paths])
print("GPU use: none (CPU-only visual QA)")

train = pd.read_csv(DATA / "train.csv")
train_series = pd.read_csv(DATA / "train_series.csv")
test = pd.read_csv(DATA / "test.csv")
test_series = pd.read_csv(DATA / "test_series.csv")
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

print()
print("train:", train.shape)
print("train_series:", train_series.shape)
print("test:", test.shape)
print("test_series:", test_series.shape)
print("sample_submission:", sample_submission.shape)

display(train.head(3))
display(train_series.head(3))

In [ ]:
### Select a report-only study with the widest plane coverage

TARGETS = [column for column in sample_submission.columns if column != "StudyInstanceUID"]
train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
train_series["StudyInstanceUID"] = train_series["StudyInstanceUID"].astype(str)
train_series["SeriesInstanceUID"] = train_series["SeriesInstanceUID"].astype(str)

report_only_ids = set(
    train.loc[train[TARGETS].isna().all(axis=1), "StudyInstanceUID"]
)
planes = {"Axial", "Coronal", "Sagittal"}
best_rows = None
best_study = None
best_score = -1

for study_uid in sorted(report_only_ids):
    group = train_series[
        (train_series["StudyInstanceUID"] == study_uid)
        & train_series["Anatomical_Plane"].isin(planes)
    ].copy()
    if group.empty:
        continue
    group = group.sort_values(
        ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"],
        ascending=[True, False, False, True],
    )
    selected = group.groupby("Anatomical_Plane", as_index=False).head(1)
    score = selected["Anatomical_Plane"].nunique()
    if score > best_score:
        best_rows, best_study, best_score = selected, study_uid, score
    if score == 3:
        break

if best_rows is None or best_score < 2:
    raise RuntimeError("Could not find a report-only study with at least two standard planes")

example_study_uid = best_study
example_series = best_rows.sort_values("Anatomical_Plane").reset_index(drop=True)
representative_rows = example_series
print("Report-only study selected; no official labels are read.")
print("Series selected by plane:")
display(example_series[["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]])


In [ ]:
%pip install -q pydicom==3.0.1

import pydicom
import matplotlib.pyplot as plt

TRAIN_DICOM_ROOT = DATA / "train_series"
assert TRAIN_DICOM_ROOT.is_dir(), TRAIN_DICOM_ROOT


In [ ]:
### Cell 5 — reconstruct and visualize one real volume

import matplotlib.pyplot as plt


def reconstruct_series(study_uid, series_uid):
  """
  Reconstruct one classic single-frame MRI series.

  Returns
  -------
  volume:
      Float32 NumPy array with axes (slice, row, column).
  geometry:
      Spatial metadata describing the reconstructed array.
  """
  folder = (
      TRAIN_DICOM_ROOT
      / str(study_uid)
      / str(series_uid)
  )

  dicom_paths = sorted(folder.glob("*.dcm"))

  if len(dicom_paths) < 2:
      raise ValueError(
          f"Expected at least two DICOM slices: {folder}"
      )

  headers = [
      pydicom.dcmread(path, stop_before_pixels=True)
      for path in dicom_paths
  ]

  reference_orientation = np.asarray(
      headers[0].ImageOrientationPatient,
      dtype=np.float64,
  )

  orientations = np.asarray(
      [
          header.ImageOrientationPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  if not np.allclose(
      orientations,
      reference_orientation,
      atol=1e-4,
      rtol=0,
  ):
      raise ValueError("Orientation varies within the series")

  pixel_spacing = np.asarray(
      headers[0].PixelSpacing,
      dtype=np.float64,
  )

  for header in headers:
      current_spacing = np.asarray(
          header.PixelSpacing,
          dtype=np.float64,
      )

      if not np.allclose(
          current_spacing,
          pixel_spacing,
          atol=1e-4,
          rtol=0,
      ):
          raise ValueError(
              "PixelSpacing varies within the series"
          )

  # DICOM IOP stores the directions across columns and rows.
  column_axis_direction = reference_orientation[:3]
  row_axis_direction = reference_orientation[3:]

  slice_normal = np.cross(
      column_axis_direction,
      row_axis_direction,
  )

  positions = np.asarray(
      [
          header.ImagePositionPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  projections = positions @ slice_normal
  physical_order = np.argsort(
      projections,
      kind="stable",
  )

  ordered_positions = positions[physical_order]
  ordered_projections = projections[physical_order]
  slice_gaps = np.diff(ordered_projections)

  if np.any(slice_gaps <= 1e-3):
      raise ValueError(
          "Repeated or indistinguishable slice positions"
      )

  slice_spacing = float(np.median(slice_gaps))

  if not np.allclose(
      slice_gaps,
      slice_spacing,
      atol=0.05,
      rtol=0.01,
  ):
      raise ValueError(
          "Irregular spacing or missing slices"
      )

  ordered_slices = []
  ordered_files = []

  for original_index in physical_order:
      path = dicom_paths[original_index]
      dcm = pydicom.dcmread(path)

      pixels = dcm.pixel_array.astype(np.float32)

      if pixels.shape != (
          int(dcm.Rows),
          int(dcm.Columns),
      ):
          raise ValueError(
              f"Unexpected pixel shape: {pixels.shape}"
          )

      # Exclude declared padding from intensity statistics.
      if hasattr(dcm, "PixelPaddingValue"):
          padding_value = float(dcm.PixelPaddingValue)

          if hasattr(dcm, "PixelPaddingRangeLimit"):
              padding_limit = float(
                  dcm.PixelPaddingRangeLimit
              )
          else:
              padding_limit = padding_value

          padding_low = min(
              padding_value,
              padding_limit,
          )
          padding_high = max(
              padding_value,
              padding_limit,
          )

          padding_mask = (
              (pixels >= padding_low)
              & (pixels <= padding_high)
          )
      else:
          padding_mask = np.zeros(
              pixels.shape,
              dtype=bool,
          )

      slope = float(
          getattr(dcm, "RescaleSlope", 1.0)
      )
      intercept = float(
          getattr(dcm, "RescaleIntercept", 0.0)
      )

      pixels = pixels * slope + intercept
      pixels[padding_mask] = np.nan

      if not np.isfinite(pixels).any():
          raise ValueError(
              f"No finite pixels in {path.name}"
          )

      ordered_slices.append(pixels)
      ordered_files.append(str(path))

  volume = np.stack(
      ordered_slices,
      axis=0,
  ).astype(np.float32)

  # Array axes are (slice, row, column).
  # This affine maps those indices into DICOM LPS millimetres.
  affine = np.eye(4, dtype=np.float64)
  affine[:3, 0] = (
      slice_normal * slice_spacing
  )
  affine[:3, 1] = (
      row_axis_direction * pixel_spacing[0]
  )
  affine[:3, 2] = (
      column_axis_direction * pixel_spacing[1]
  )
  affine[:3, 3] = ordered_positions[0]

  geometry = {
      "array_axes": ("slice", "row", "column"),
      "coordinate_system": "DICOM LPS millimetres",
      "spacing_drc_mm": (
          slice_spacing,
          float(pixel_spacing[0]),
          float(pixel_spacing[1]),
      ),
      "affine_array_to_lps": affine,
      "slice_normal_lps": slice_normal,
      "ordered_positions_lps": ordered_positions,
      "ordered_files": ordered_files,
      "photometric_interpretation": str(
          getattr(
              headers[0],
              "PhotometricInterpretation",
              "",
          )
      ),
  }

  return volume, geometry




In [ ]:
### Cell 6 — compare sagittal, coronal, and axial reconstructions

# Prefer a fluid-sensitive series for each anatomical plane.
representative_rows = (
  example_series
  .sort_values(
      ["Anatomical_Plane", "Fluid_Sensitive"],
      ascending=[True, False],
  )
  .groupby("Anatomical_Plane", as_index=False)
  .head(1)
  .sort_values("Anatomical_Plane")
  .reset_index(drop=True)
)

fig, axes = plt.subplots(
  len(representative_rows),
  3,
  figsize=(13, 4 * len(representative_rows)),
)

reconstruction_summary = []

for row_index, series_row in representative_rows.iterrows():
  plane = series_row["Anatomical_Plane"]

  plane_volume, plane_geometry = reconstruct_series(
      series_row["StudyInstanceUID"],
      series_row["SeriesInstanceUID"],
  )

  (
      plane_slice_spacing,
      plane_row_spacing,
      plane_column_spacing,
  ) = plane_geometry["spacing_drc_mm"]

  finite = plane_volume[
      np.isfinite(plane_volume)
  ]

  window_low, window_high = np.percentile(
      finite,
      [1, 99],
  )

  middle_slice = plane_volume.shape[0] // 2
  middle_row = plane_volume.shape[1] // 2
  middle_column = plane_volume.shape[2] // 2

  # Native acquisition plane.
  axes[row_index, 0].imshow(
      plane_volume[middle_slice],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_row_spacing
          / plane_column_spacing
      ),
  )
  axes[row_index, 0].set_title(
      f"{plane}: native centre slice"
  )
  axes[row_index, 0].set_xlabel("column")
  axes[row_index, 0].set_ylabel("row")

  # Orthogonal centre-row view.
  axes[row_index, 1].imshow(
      plane_volume[:, middle_row, :],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_slice_spacing
          / plane_column_spacing
      ),
  )
  axes[row_index, 1].set_title(
      f"{plane}: centre row"
  )
  axes[row_index, 1].set_xlabel("column")
  axes[row_index, 1].set_ylabel(
      "physically ordered slice"
  )

  # Orthogonal centre-column view.
  axes[row_index, 2].imshow(
      plane_volume[:, :, middle_column],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_slice_spacing
          / plane_row_spacing
      ),
  )
  axes[row_index, 2].set_title(
      f"{plane}: centre column"
  )
  axes[row_index, 2].set_xlabel("row")
  axes[row_index, 2].set_ylabel(
      "physically ordered slice"
  )

  reconstruction_summary.append(
      {
          "plane": plane,
          "fluid_sensitive":
              series_row["Fluid_Sensitive"],
          "fat_suppression":
              series_row["Fat_Suppression"],
          "shape_drc": tuple(
              plane_volume.shape
          ),
          "spacing_drc_mm":
              plane_geometry["spacing_drc_mm"],
          "physical_fov_mm": tuple(
              round(size * spacing, 2)
              for size, spacing in zip(
                  plane_volume.shape,
                  plane_geometry[
                      "spacing_drc_mm"
                  ],
              )
          ),
      }
  )

plt.tight_layout()
plt.show()

display(
  pd.DataFrame(reconstruction_summary)
)

## Visual review gate

Inspect the reconstruction figure in the preceding cell before continuing. If any view is blank, truncated, or incorrectly oriented, stop and fix preprocessing first. Run the remaining cells individually after confirming the images are usable. The selected study is report-only; the notebook never reads official labels.
